#  Contextual Retrieval

1. Contextual Retrieval의 개념과 기존 RAG의 한계 이해
2. LLM을 활용한 청크별 컨텍스트 생성 방법 습득
3. BM25 + Contextual Embedding 하이브리드 검색 구현
4. Contextual Retrieval의 성능 향상 효과 측정

---

## 환경 설정 및 준비

`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [2]:
import os
from pprint import pprint
from typing import List, Tuple

`(3) langfuse handler 설정`

In [3]:
from langfuse.langchain import CallbackHandler

# LangChain 콜백 핸들러 생성
langfuse_handler = CallbackHandler()

---

## 문제 제기: 청킹 시 컨텍스트 손실

### 기존 RAG의 한계

일반적인 RAG 시스템에서는 문서를 작은 청크로 분할하여 검색합니다. 하지만 이 과정에서 **중요한 맥락 정보가 손실**될 수 있습니다.

```mermaid
graph TD
    A["원본 문서<br/>테슬라 2023 Q3 실적 보고서"] -->|청킹| B["청크 50<br/>이 분기의 매출은 전년 동기 대비 8% 증가..."]
    B -->|❌ 기존 RAG| C["맥락 손실<br/>어떤 회사? 언제?"]
    A --> D[문서 전체 맥락]
    B --> E[컨텍스트 추가]
    D --> E
    E -->|✅ Contextual Retrieval| F["컨텍스트 + 청크<br/>테슬라 2023 Q3 매출 성과..."]
```

### 예시

원본 문서:
```
테슬라의 2023년 3분기 실적 보고서
...
[청크 50] 이 분기의 매출은 전년 동기 대비 8% 증가한 233억 달러를 기록했습니다.
```

**문제점**: 청크 50만 보면 "어떤 회사", "언제" 등의 맥락을 알 수 없습니다.

### Anthropic의 해결책: Contextual Retrieval

**핵심 아이디어**: 각 청크에 **문서 전체 맥락을 설명하는 컨텍스트를 추가**합니다.

```
[컨텍스트] 이 청크는 테슬라의 2023년 3분기 실적 보고서에서 매출 성과를 다룹니다.
[원본 청크] 이 분기의 매출은 전년 동기 대비 8% 증가한 233억 달러를 기록했습니다.
```

### 성능 향상 효과 (Anthropic 연구)

| 기법 | 검색 실패율 감소 |
|------|----------------|
| Contextual Embedding | 35% |
| + Contextual BM25 | 49% |
| + Reranker | 67% |


---

## 1. 데이터 준비

### 1-1) 샘플 문서 로드

In [13]:
from langchain_core.documents import Document

# 긴 샘플 문서 (실제 사용 시에는 파일에서 로드)
sample_document = """
# 테슬라 2023년 연간 보고서

## 1. 회사 개요

테슬라(Tesla, Inc.)는 2003년 설립된 미국의 전기자동차 및 청정에너지 회사입니다. 
본사는 텍사스 오스틴에 위치해 있으며, 일론 머스크가 CEO로 재직 중입니다.
테슬라는 전기차, 에너지 저장 시스템, 태양광 패널 등을 제조 및 판매합니다.

## 2. 2023년 실적 요약

2023년 테슬라의 총 매출은 967억 달러를 기록했습니다.
전년 대비 19% 증가한 수치입니다.
자동차 부문 매출이 824억 달러로 전체의 85%를 차지했습니다.
에너지 저장 부문은 60억 달러의 매출을 달성했습니다.

## 3. 생산 및 판매

2023년 테슬라는 총 184만 5,985대의 차량을 생산했습니다.
이 중 Model Y가 120만 대 이상으로 가장 많이 생산되었습니다.
Model 3는 약 50만 대가 생산되어 두 번째로 많았습니다.
전 세계 인도량은 180만 8,581대를 기록했습니다.

## 4. 기술 개발

### 4.1 자율주행
FSD(Full Self-Driving) 베타 버전이 북미 전역으로 확대되었습니다.
누적 FSD 주행 거리가 10억 마일을 돌파했습니다.
Dojo 슈퍼컴퓨터를 활용한 AI 학습이 진행 중입니다.

### 4.2 배터리 기술
4680 배터리 셀의 대량 생산이 시작되었습니다.
텍사스 기가팩토리에서 주당 1,000만 셀 이상을 생산하고 있습니다.
배터리 비용을 kWh당 100달러 미만으로 낮추는 것이 목표입니다.

## 5. 미래 전망

2024년에는 차세대 저가 모델 출시가 예정되어 있습니다.
사이버트럭의 본격적인 양산도 시작될 예정입니다.
테슬라 로보택시 서비스 출시를 위한 준비가 진행 중입니다.
"""

# Document 객체로 변환
doc = Document(
    page_content=sample_document,
    metadata={"source": "tesla_annual_report_2023.md", "year": 2023}
)

print(f"문서 길이: {len(doc.page_content)} 글자")

문서 길이: 844 글자


### 1-2) 문서 청킹

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할기 설정
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,      # 작은 청크로 분할
    chunk_overlap=50,    # 50자 중복
    separators=["\n\n", "\n", ". ",],
)

# 문서 분할
chunks = text_splitter.split_documents([doc])

print(f"생성된 청크 수: {len(chunks)}")
print("="*80)

for i, chunk in enumerate(chunks[:5]):
    print(f"\n[청크 {i+1}] ({len(chunk.page_content)}자)")
    print("-"*40)
    print(chunk.page_content[:200] + "..." if len(chunk.page_content) > 200 else chunk.page_content)

생성된 청크 수: 6

[청크 1] (193자)
----------------------------------------
# 테슬라 2023년 연간 보고서

## 1. 회사 개요

테슬라(Tesla, Inc.)는 2003년 설립된 미국의 전기자동차 및 청정에너지 회사입니다. 
본사는 텍사스 오스틴에 위치해 있으며, 일론 머스크가 CEO로 재직 중입니다.
테슬라는 전기차, 에너지 저장 시스템, 태양광 패널 등을 제조 및 판매합니다.

## 2. 2023년 실적 요약

[청크 2] (156자)
----------------------------------------
## 2. 2023년 실적 요약

2023년 테슬라의 총 매출은 967억 달러를 기록했습니다.
전년 대비 19% 증가한 수치입니다.
자동차 부문 매출이 824억 달러로 전체의 85%를 차지했습니다.
에너지 저장 부문은 60억 달러의 매출을 달성했습니다.

## 3. 생산 및 판매

[청크 3] (172자)
----------------------------------------
## 3. 생산 및 판매

2023년 테슬라는 총 184만 5,985대의 차량을 생산했습니다.
이 중 Model Y가 120만 대 이상으로 가장 많이 생산되었습니다.
Model 3는 약 50만 대가 생산되어 두 번째로 많았습니다.
전 세계 인도량은 180만 8,581대를 기록했습니다.

## 4. 기술 개발

[청크 4] (134자)
----------------------------------------
## 4. 기술 개발

### 4.1 자율주행
FSD(Full Self-Driving) 베타 버전이 북미 전역으로 확대되었습니다.
누적 FSD 주행 거리가 10억 마일을 돌파했습니다.
Dojo 슈퍼컴퓨터를 활용한 AI 학습이 진행 중입니다.

[청크 5] (132자)
----------------------------------------
### 4.2 배터리 기술
4680 배터리 셀의 대량 생산이 시작되었습니다.
텍사스 기가

---

## 2. 컨텍스트 생성

### Contextual Retrieval의 핵심

LLM을 사용하여 각 청크에 대한 **간결한 맥락 설명**을 생성합니다.

### 2-1) 컨텍스트 생성 프롬프트

In [15]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# LLM 초기화 (컨텍스트 생성용 - 가벼운 모델 사용)
context_llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)

# Anthropic의 컨텍스트 생성 프롬프트 (한국어 버전)
context_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 문서의 청크에 맥락을 추가하는 전문가입니다.
주어진 청크가 전체 문서에서 어떤 위치에 있고 무엇에 대한 내용인지 간결하게 설명하세요.
설명은 50-100자 이내로 작성하세요.
오직 맥락 설명만 출력하세요."""),
    ("user", """<document>
{whole_document}
</document>

위 문서에서 아래 청크의 맥락을 설명해주세요:

<chunk>
{chunk_content}
</chunk>

맥락 설명:""")
])

# 컨텍스트 생성 체인
context_chain = context_prompt | context_llm | StrOutputParser()

print("컨텍스트 생성 체인 준비 완료")

컨텍스트 생성 체인 준비 완료


### 2-2) 배치 컨텍스트 생성

In [16]:
# 각 청크에 대해 컨텍스트 생성
def generate_contexts(chunks: List[Document], whole_document: str) -> List[str]:
    """청크들에 대한 컨텍스트를 배치로 생성합니다."""
    inputs = [
        {"whole_document": whole_document, "chunk_content": chunk.page_content}
        for chunk in chunks
    ]
    
    # 배치 처리로 효율적으로 생성
    contexts = context_chain.batch(
        inputs, 
        config={"max_concurrency": 5, "callbacks": [langfuse_handler]}
    )
    
    return contexts

# 컨텍스트 생성 실행
print("컨텍스트 생성 중...")
contexts = generate_contexts(chunks, sample_document)

print(f"\n생성된 컨텍스트 수: {len(contexts)}")
print("="*80)

for i, (chunk, context) in enumerate(zip(chunks[:5], contexts[:5])):
    print(f"\n[청크 {i+1}]")
    print(f"원본: {chunk.page_content[:100]}...")
    print(f"컨텍스트: {context}")

컨텍스트 생성 중...

생성된 컨텍스트 수: 6

[청크 1]
원본: # 테슬라 2023년 연간 보고서

## 1. 회사 개요

테슬라(Tesla, Inc.)는 2003년 설립된 미국의 전기자동차 및 청정에너지 회사입니다. 
본사는 텍사스 오스틴에 ...
컨텍스트: 이 청크는 테슬라의 회사 개요와 2023년 실적 요약에 대한 내용을 소개하며, 전체 문서의 서론 부분으로 회사의 기본 정보와 연간 실적 개요를 설명합니다.

[청크 2]
원본: ## 2. 2023년 실적 요약

2023년 테슬라의 총 매출은 967억 달러를 기록했습니다.
전년 대비 19% 증가한 수치입니다.
자동차 부문 매출이 824억 달러로 전체의 85...
컨텍스트: 2023년 테슬라의 재무 성과와 매출, 생산량, 판매량에 대한 핵심 실적 요약으로, 전체 실적과 생산/판매 현황의 연결된 내용을 포함합니다.

[청크 3]
원본: ## 3. 생산 및 판매

2023년 테슬라는 총 184만 5,985대의 차량을 생산했습니다.
이 중 Model Y가 120만 대 이상으로 가장 많이 생산되었습니다.
Model 3...
컨텍스트: 이 청크는 2023년 테슬라의 차량 생산 및 판매 실적과 기술 개발 현황을 다루며, 생산량과 모델별 판매량을 소개하고 있습니다.

[청크 4]
원본: ## 4. 기술 개발

### 4.1 자율주행
FSD(Full Self-Driving) 베타 버전이 북미 전역으로 확대되었습니다.
누적 FSD 주행 거리가 10억 마일을 돌파했습니...
컨텍스트: 이 청크는 2023년 테슬라의 기술 개발 부문, 특히 자율주행과 AI 학습 관련 최신 성과와 진행 상황을 설명하는 부분입니다.

[청크 5]
원본: ### 4.2 배터리 기술
4680 배터리 셀의 대량 생산이 시작되었습니다.
텍사스 기가팩토리에서 주당 1,000만 셀 이상을 생산하고 있습니다.
배터리 비용을 kWh당 100달러...
컨텍스트: 이 청크는 2023년 테슬라의 배터리 기술 개발 현황과 생산 확대, 비용 절감 목표를 다루며, 

### 2-3) Contextual 청크 생성

In [17]:
# 컨텍스트가 추가된 청크 생성
contextual_chunks = []

for i, (chunk, context) in enumerate(zip(chunks, contexts)):
    # 컨텍스트 + 원본 청크 결합
    contextual_content = f"[맥락] {context}\n\n{chunk.page_content}"
    
    contextual_chunk = Document(
        page_content=contextual_content,
        metadata={
            **chunk.metadata,
            "chunk_id": i,
            "original_content": chunk.page_content,
            "context": context,
        }
    )
    contextual_chunks.append(contextual_chunk)

print(f"Contextual 청크 생성 완료: {len(contextual_chunks)}개")
print("="*80)

# 예시 출력
print("\n[Contextual 청크 예시]")
print(contextual_chunks[3].page_content)

Contextual 청크 생성 완료: 6개

[Contextual 청크 예시]
[맥락] 이 청크는 2023년 테슬라의 기술 개발 부문, 특히 자율주행과 AI 학습 관련 최신 성과와 진행 상황을 설명하는 부분입니다.

## 4. 기술 개발

### 4.1 자율주행
FSD(Full Self-Driving) 베타 버전이 북미 전역으로 확대되었습니다.
누적 FSD 주행 거리가 10억 마일을 돌파했습니다.
Dojo 슈퍼컴퓨터를 활용한 AI 학습이 진행 중입니다.


---

## 3. Contextual Embedding 검색

컨텍스트가 추가된 청크를 임베딩하여 검색합니다.

### 3-1) 벡터 저장소 생성

In [18]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# 임베딩 모델 초기화
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 일반 청크 벡터 저장소 (비교용)
normal_vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="normal_chunks",
    persist_directory="./chroma_db"
)

# Contextual 청크 벡터 저장소
contextual_vectorstore = Chroma.from_documents(
    documents=contextual_chunks,
    embedding=embeddings,
    collection_name="contextual_chunks",
    persist_directory="./chroma_db"
)

print("벡터 저장소 생성 완료")
print(f"- 일반 청크: {normal_vectorstore._collection.count()}개")
print(f"- Contextual 청크: {contextual_vectorstore._collection.count()}개")

벡터 저장소 생성 완료
- 일반 청크: 12개
- Contextual 청크: 12개


### 3-2) 검색 비교 테스트

In [19]:
# 테스트 쿼리
test_queries = [
    "테슬라의 2023년 총 매출은 얼마인가?",
    "Model Y 생산량은 몇 대인가?",
    "4680 배터리 셀 생산량은?",
]

# Retriever 생성
normal_retriever = normal_vectorstore.as_retriever(search_kwargs={"k": 3})
contextual_retriever = contextual_vectorstore.as_retriever(search_kwargs={"k": 3})

for query in test_queries:
    print(f"\n{'='*80}")
    print(f"쿼리: '{query}'")
    print("="*80)
    
    # 일반 검색
    normal_results = normal_retriever.invoke(query)
    print(f"\n[일반 검색 결과]")
    for i, doc in enumerate(normal_results):
        print(f"  {i+1}. {doc.page_content[:100]}...")
    
    # Contextual 검색
    contextual_results = contextual_retriever.invoke(query)
    print(f"\n[Contextual 검색 결과]")
    for i, doc in enumerate(contextual_results):
        print(f"  {i+1}. {doc.page_content[:100]}...")


쿼리: '테슬라의 2023년 총 매출은 얼마인가?'

[일반 검색 결과]
  1. ## 2. 2023년 실적 요약

2023년 테슬라의 총 매출은 967억 달러를 기록했습니다.
전년 대비 19% 증가한 수치입니다.
자동차 부문 매출이 824억 달러로 전체의 85...
  2. ## 2. 2023년 실적 요약

2023년 테슬라의 총 매출은 967억 달러를 기록했습니다.
전년 대비 19% 증가한 수치입니다.
자동차 부문 매출이 824억 달러로 전체의 85...
  3. # 테슬라 2023년 연간 보고서

## 1. 회사 개요

테슬라(Tesla, Inc.)는 2003년 설립된 미국의 전기자동차 및 청정에너지 회사입니다. 
본사는 텍사스 오스틴에 ...

[Contextual 검색 결과]
  1. [맥락] 2023년 테슬라의 재무 성과와 매출, 생산량, 판매량에 대한 핵심 실적 요약으로, 전체 실적과 생산/판매 현황의 연결된 내용을 포함합니다.

## 2. 2023년 실적 ...
  2. [맥락] 2023년 테슬라의 재무 성과와 매출, 생산량, 판매량에 대한 핵심 실적 요약으로, 전체 실적과 생산/판매 현황의 연결된 내용을 포함합니다.

## 2. 2023년 실적 ...
  3. [맥락] 이 청크는 2023년 테슬라의 차량 생산 및 판매 실적과 기술 개발 현황을 다루며, 생산량과 모델별 인기도를 소개합니다.

## 3. 생산 및 판매

2023년 테슬라는 ...

쿼리: 'Model Y 생산량은 몇 대인가?'

[일반 검색 결과]
  1. ## 3. 생산 및 판매

2023년 테슬라는 총 184만 5,985대의 차량을 생산했습니다.
이 중 Model Y가 120만 대 이상으로 가장 많이 생산되었습니다.
Model 3...
  2. ## 3. 생산 및 판매

2023년 테슬라는 총 184만 5,985대의 차량을 생산했습니다.
이 중 Model Y가 120만 대 이상으로 가장 많이 생산되었습니다.
Model 3...
  3. ### 4.2 배터리 기술
4680 배터

---

## 4. Contextual BM25 검색

BM25는 정확한 키워드 매칭에 강점이 있습니다. 특히 **고유명사, 숫자, 에러 코드** 등의 검색에 효과적입니다.

### 4-1) BM25 Retriever 설정

In [20]:
from langchain_community.retrievers import BM25Retriever
from kiwipiepy import Kiwi

# Kiwi 한국어 형태소 분석기 초기화
kiwi = Kiwi()

# 사용자 정의 단어 추가 (고유명사)
kiwi.add_user_word('테슬라', 'NNP')

# 한국어 토크나이저 전처리 함수
def kiwi_preprocess_func(text):
    """Kiwi 형태소 분석기를 사용한 토큰화"""
    return [t.form for t in kiwi.tokenize(text)]

# 일반 BM25 Retriever (Kiwi 토크나이저 적용)
normal_bm25 = BM25Retriever.from_documents(
    documents=chunks,
    preprocess_func=kiwi_preprocess_func,
    k=3
)

# Contextual BM25 Retriever (Kiwi 토크나이저 적용)
contextual_bm25 = BM25Retriever.from_documents(
    documents=contextual_chunks,
    preprocess_func=kiwi_preprocess_func,
    k=3
)

print("BM25 Retriever 생성 완료 (Kiwi 토크나이저 적용)")

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_24860\2643254040.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


BM25 Retriever 생성 완료 (Kiwi 토크나이저 적용)


### 4-2) BM25 검색 비교

In [21]:
# 정확한 키워드가 필요한 쿼리
keyword_queries = [
    "967억 달러",
    "FSD 베타",
    "Dojo 슈퍼컴퓨터",
]

for query in keyword_queries:
    print(f"\n{'='*80}")
    print(f"쿼리: '{query}'")
    print("="*80)
    
    # 일반 BM25
    normal_results = normal_bm25.invoke(query)
    print(f"\n[일반 BM25]")
    for i, doc in enumerate(normal_results):
        print(f"  {i+1}. {doc.page_content[:100]}...")
    
    # Contextual BM25
    contextual_results = contextual_bm25.invoke(query)
    print(f"\n[Contextual BM25]")
    for i, doc in enumerate(contextual_results):
        print(f"  {i+1}. {doc.page_content[:100]}...")


쿼리: '967억 달러'

[일반 BM25]
  1. ## 2. 2023년 실적 요약

2023년 테슬라의 총 매출은 967억 달러를 기록했습니다.
전년 대비 19% 증가한 수치입니다.
자동차 부문 매출이 824억 달러로 전체의 85...
  2. ### 4.2 배터리 기술
4680 배터리 셀의 대량 생산이 시작되었습니다.
텍사스 기가팩토리에서 주당 1,000만 셀 이상을 생산하고 있습니다.
배터리 비용을 kWh당 100달러...
  3. ## 4. 기술 개발

### 4.1 자율주행
FSD(Full Self-Driving) 베타 버전이 북미 전역으로 확대되었습니다.
누적 FSD 주행 거리가 10억 마일을 돌파했습니...

[Contextual BM25]
  1. [맥락] 2023년 테슬라의 재무 성과와 매출, 생산량, 판매량에 대한 핵심 실적 요약으로, 전체 실적과 생산/판매 현황의 연결된 내용을 포함합니다.

## 2. 2023년 실적 ...
  2. [맥락] 이 청크는 2023년 테슬라의 기술 개발 부문, 특히 자율주행과 AI 학습 관련 최신 성과와 진행 상황을 설명하는 부분입니다.

## 4. 기술 개발

### 4.1 자율...
  3. [맥락] 이 청크는 2023년 테슬라의 배터리 기술 개발 현황과 생산 확대, 비용 절감 목표를 다루며, 기술 혁신과 미래 전략의 핵심 내용을 포함합니다.

### 4.2 배터리 기...

쿼리: 'FSD 베타'

[일반 BM25]
  1. ## 4. 기술 개발

### 4.1 자율주행
FSD(Full Self-Driving) 베타 버전이 북미 전역으로 확대되었습니다.
누적 FSD 주행 거리가 10억 마일을 돌파했습니...
  2. ## 5. 미래 전망

2024년에는 차세대 저가 모델 출시가 예정되어 있습니다.
사이버트럭의 본격적인 양산도 시작될 예정입니다.
테슬라 로보택시 서비스 출시를 위한 준비가 진행 ...
  3. ### 4.2 배터리 기술
4680 배터리 셀의 대량 생산이 시작되었습니다.
텍사스 기가팩토리에

---

## 5. 하이브리드 검색 (Embedding + BM25)

**의미 기반 검색(Embedding)** 과 **키워드 기반 검색(BM25)** 을 결합하여 최상의 결과를 얻습니다.

### 5-1) EnsembleRetriever 설정

In [22]:
from langchain_classic.retrievers import EnsembleRetriever

# 일반 하이브리드 Retriever
normal_hybrid = EnsembleRetriever(
    retrievers=[normal_retriever, normal_bm25],
    weights=[0.5, 0.5],  # Embedding과 BM25 동일 가중치
)

# Contextual 하이브리드 Retriever
contextual_hybrid = EnsembleRetriever(
    retrievers=[contextual_retriever, contextual_bm25],
    weights=[0.5, 0.5],
)

print("하이브리드 Retriever 생성 완료")

하이브리드 Retriever 생성 완료


### 5-2) 하이브리드 검색 테스트

In [23]:
# 다양한 유형의 쿼리
hybrid_queries = [
    "테슬라의 연간 매출 실적은?",           # 의미 기반
    "4680 배터리 kWh당 목표 가격",          # 키워드 기반
    "2024년 출시 예정인 새로운 모델은?",     # 혼합
]

for query in hybrid_queries:
    print(f"\n{'='*80}")
    print(f"쿼리: '{query}'")
    print("="*80)
    
    # 일반 하이브리드
    normal_results = normal_hybrid.invoke(query)
    print(f"\n[일반 하이브리드] 결과 {len(normal_results)}개")
    for i, doc in enumerate(normal_results[:2]):
        print(f"  {i+1}. {doc.page_content[:80]}...")
    
    # Contextual 하이브리드
    contextual_results = contextual_hybrid.invoke(query)
    print(f"\n[Contextual 하이브리드] 결과 {len(contextual_results)}개")
    for i, doc in enumerate(contextual_results[:2]):
        # 원본 내용 표시
        original = doc.metadata.get('original_content', doc.page_content[:80])
        print(f"  {i+1}. {original[:80]}...")


쿼리: '테슬라의 연간 매출 실적은?'

[일반 하이브리드] 결과 3개
  1. ## 2. 2023년 실적 요약

2023년 테슬라의 총 매출은 967억 달러를 기록했습니다.
전년 대비 19% 증가한 수치입니다.
자동차 부문...
  2. # 테슬라 2023년 연간 보고서

## 1. 회사 개요

테슬라(Tesla, Inc.)는 2003년 설립된 미국의 전기자동차 및 청정에너지 회...

[Contextual 하이브리드] 결과 3개
  1. ## 2. 2023년 실적 요약

2023년 테슬라의 총 매출은 967억 달러를 기록했습니다.
전년 대비 19% 증가한 수치입니다.
자동차 부문...
  2. # 테슬라 2023년 연간 보고서

## 1. 회사 개요

테슬라(Tesla, Inc.)는 2003년 설립된 미국의 전기자동차 및 청정에너지 회...

쿼리: '4680 배터리 kWh당 목표 가격'

[일반 하이브리드] 결과 4개
  1. ### 4.2 배터리 기술
4680 배터리 셀의 대량 생산이 시작되었습니다.
텍사스 기가팩토리에서 주당 1,000만 셀 이상을 생산하고 있습니다...
  2. ## 5. 미래 전망

2024년에는 차세대 저가 모델 출시가 예정되어 있습니다.
사이버트럭의 본격적인 양산도 시작될 예정입니다.
테슬라 로보택...

[Contextual 하이브리드] 결과 5개
  1. ### 4.2 배터리 기술
4680 배터리 셀의 대량 생산이 시작되었습니다.
텍사스 기가팩토리에서 주당 1,000만 셀 이상을 생산하고 있습니다...
  2. ### 4.2 배터리 기술
4680 배터리 셀의 대량 생산이 시작되었습니다.
텍사스 기가팩토리에서 주당 1,000만 셀 이상을 생산하고 있습니다...

쿼리: '2024년 출시 예정인 새로운 모델은?'

[일반 하이브리드] 결과 3개
  1. ## 5. 미래 전망

2024년에는 차세대 저가 모델 출시가 예정되어 있습니다.
사이버트럭의 본격적인 양산도 시작될 예정입니다.
테슬라 로보택...
  2. ## 3. 생산 

---

## 6. Reranker 추가

하이브리드 검색 결과를 **Reranker로 재순위화**하여 최종 성능을 극대화합니다.

### 6-1) Reranker 설정

In [24]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# Cross-Encoder 모델 초기화
cross_encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
reranker = CrossEncoderReranker(model=cross_encoder, top_n=3)

# Contextual Hybrid + Reranker
contextual_rerank_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=contextual_hybrid,
)

print("Reranker 설정 완료")

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 5210.55it/s]


Reranker 설정 완료


### 6-2) 최종 검색 테스트

In [25]:
# 최종 검색 테스트
final_queries = [
    "테슬라의 자율주행 기술 현황은?",
    "2023년 차량 생산량과 인도량은?",
    "배터리 비용 절감 목표는?",
]

for query in final_queries:
    print(f"\n{'='*80}")
    print(f"쿼리: '{query}'")
    print("="*80)
    
    # Contextual Hybrid + Reranker
    results = contextual_rerank_retriever.invoke(query, config={"callbacks": [langfuse_handler]})
    
    print(f"\n[Contextual Hybrid + Reranker] Top {len(results)}개")
    for i, doc in enumerate(results):
        original = doc.metadata.get('original_content', doc.page_content)
        context = doc.metadata.get('context', 'N/A')
        print(f"\n  [{i+1}] 맥락: {context}")
        print(f"      내용: {original[:100]}...")


쿼리: '테슬라의 자율주행 기술 현황은?'

[Contextual Hybrid + Reranker] Top 3개

  [1] 맥락: 이 청크는 2023년 테슬라의 기술 개발 부문, 특히 자율주행과 AI 학습 관련 최신 성과와 진행 상황을 설명하는 부분입니다.
      내용: ## 4. 기술 개발

### 4.1 자율주행
FSD(Full Self-Driving) 베타 버전이 북미 전역으로 확대되었습니다.
누적 FSD 주행 거리가 10억 마일을 돌파했습니...

  [2] 맥락: 이 청크는 2023년 테슬라의 차량 생산 및 판매 실적과 기술 개발 현황을 다루며, 생산량과 모델별 판매량을 소개하고 있습니다.
      내용: ## 3. 생산 및 판매

2023년 테슬라는 총 184만 5,985대의 차량을 생산했습니다.
이 중 Model Y가 120만 대 이상으로 가장 많이 생산되었습니다.
Model 3...

  [3] 맥락: 2023년 테슬라의 재무 성과와 매출, 생산량, 판매량에 대한 핵심 실적 요약으로, 전체 실적과 생산/판매 현황의 연결된 내용을 포함합니다.
      내용: ## 2. 2023년 실적 요약

2023년 테슬라의 총 매출은 967억 달러를 기록했습니다.
전년 대비 19% 증가한 수치입니다.
자동차 부문 매출이 824억 달러로 전체의 85...

쿼리: '2023년 차량 생산량과 인도량은?'

[Contextual Hybrid + Reranker] Top 3개

  [1] 맥락: 이 청크는 2023년 테슬라의 차량 생산 및 판매 실적과 기술 개발 현황을 다루며, 생산량과 모델별 판매량을 소개하고 있습니다.
      내용: ## 3. 생산 및 판매

2023년 테슬라는 총 184만 5,985대의 차량을 생산했습니다.
이 중 Model Y가 120만 대 이상으로 가장 많이 생산되었습니다.
Model 3...

  [2] 맥락: 이 청크는 2023년 테슬라의 차량 생산 및 판매 실적과 기술 개발 현황을 다루며, 생산량과 모델별 인기도를 소개합니

---

## 7. RAG 체인 구성

Contextual Retrieval을 활용한 완전한 RAG 시스템을 구축합니다.

In [26]:
from langchain_core.runnables import RunnablePassthrough

# 답변 생성용 LLM
answer_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# RAG 프롬프트
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 문서 기반 질의응답 전문가입니다.
주어진 문맥을 바탕으로 질문에 정확하게 답변하세요.
문맥에 없는 내용은 "문서에서 해당 정보를 찾을 수 없습니다."라고 답변하세요."""),
    ("user", """문맥:
{context}

질문: {question}

답변:""")
])

# 문서 포맷팅 함수
def format_docs(docs):
    formatted = []
    for doc in docs:
        # 원본 내용과 맥락 모두 포함
        original = doc.metadata.get('original_content', doc.page_content)
        context = doc.metadata.get('context', '')
        if context:
            formatted.append(f"[맥락: {context}]\n{original}")
        else:
            formatted.append(original)
    return "\n\n---\n\n".join(formatted)

# RAG 체인 구성
rag_chain = (
    {"context": contextual_rerank_retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | answer_llm
    | StrOutputParser()
)

print("RAG 체인 구성 완료")

RAG 체인 구성 완료


In [27]:
# RAG 체인 실행
questions = [
    "테슬라의 2023년 매출 구성을 설명해주세요.",
    "FSD의 현재 상황은 어떤가요?",
    "2024년에 예정된 신제품은 무엇인가요?",
    "4680 배터리의 생산 현황과 목표는?",
]

for question in questions:
    print(f"\n{'='*80}")
    print(f"Q: {question}")
    print("-"*80)
    
    answer = rag_chain.invoke(question, config={"callbacks": [langfuse_handler]})
    print(f"A: {answer}")


Q: 테슬라의 2023년 매출 구성을 설명해주세요.
--------------------------------------------------------------------------------
A: 테슬라의 2023년 총 매출은 967억 달러이며, 이 중 자동차 부문 매출이 824억 달러로 전체 매출의 85%를 차지했습니다. 에너지 저장 부문은 60억 달러의 매출을 기록했습니다.

Q: FSD의 현재 상황은 어떤가요?
--------------------------------------------------------------------------------
A: FSD(Full Self-Driving) 베타 버전이 북미 전역으로 확대되었으며, 누적 FSD 주행 거리가 10억 마일을 돌파했습니다. 또한, Dojo 슈퍼컴퓨터를 활용한 AI 학습이 진행 중입니다.

Q: 2024년에 예정된 신제품은 무엇인가요?
--------------------------------------------------------------------------------
A: 2024년에 예정된 신제품은 차세대 저가 모델과 사이버트럭입니다.

Q: 4680 배터리의 생산 현황과 목표는?
--------------------------------------------------------------------------------
A: 4680 배터리 셀의 대량 생산이 시작되었으며, 텍사스 기가팩토리에서 주당 1,000만 셀 이상을 생산하고 있습니다. 또한, 배터리 비용을 kWh당 100달러 미만으로 낮추는 것이 목표입니다.


---

## 8. 성능 비교 요약

In [28]:
# 다양한 Retriever 성능 비교 (Kiwi 기반)
from kiwipiepy import Kiwi

retrievers_to_compare = {
    "일반 Embedding": normal_retriever,
    "일반 BM25": normal_bm25,
    "일반 Hybrid": normal_hybrid,
    "Contextual Embedding": contextual_retriever,
    "Contextual BM25": contextual_bm25,
    "Contextual Hybrid": contextual_hybrid,
    "Contextual + Reranker": contextual_rerank_retriever,
}

# 테스트 쿼리와 기대 키워드
# 핵심: 컨텍스트에만 있고 원본 청크에는 없는 키워드로 검색
# k=1로 설정하여 정확한 청크 매칭 테스트
test_queries = [
    # 직접적인 쿼리 (일반 검색도 가능) - 원본 청크에 키워드 존재
    {"query": "테슬라의 2023년 자동차 부문 매출은?", "expected": "824억 달러"},
    {"query": "4680 배터리 생산량은?", "expected": "1,000만 셀"},
    
    # 맥락 의존적 쿼리 - 컨텍스트에만 있는 표현 사용
    # 원본 청크: "FSD 베타 버전이... Dojo 슈퍼컴퓨터를 활용한 AI 학습"
    # 컨텍스트: "기술 개발 부문, 자율주행과 AI 학습 관련 최신 성과"
    {"query": "테슬라의 최신 성과와 진행 상황", "expected": "Dojo"},
    
    # 원본 청크: "4680 배터리 셀의 대량 생산... kWh당 100달러"
    # 컨텍스트: "기술 혁신과 미래 전략의 핵심 내용"
    {"query": "테슬라의 기술 혁신 핵심 내용", "expected": "100달러"},
    
    # 원본 청크: "2023년 테슬라의 총 매출은 967억 달러... 에너지 저장 부문"
    # 컨텍스트: "재무 성과와 매출, 생산량에 대한 핵심 내용을 요약"
    {"query": "테슬라 재무 성과 요약", "expected": "967억 달러"},
    
    # 원본 청크: "Tesla, Inc.는 2003년 설립... 일론 머스크가 CEO"
    # 컨텍스트: "회사의 설립 배경, 주요 사업 분야"
    {"query": "테슬라 설립 배경과 사업 분야", "expected": "2003년"},
]

kiwi = Kiwi()

def evaluate_contextual_retrievers(retrievers: dict, test_queries: list, k: int = 1):
    """Contextual vs Non-Contextual Retriever 성능 비교
    
    k=1로 설정하여 가장 관련성 높은 1개 청크만 검색.
    문서가 작을 때 (6개 청크) k=3이면 50%를 검색하므로 변별력이 낮음.
    """
    results_dict = {}
    
    for name, retriever in retrievers.items():
        correct = 0
        total = len(test_queries)
        
        for test in test_queries:
            query = test["query"]
            expected = test["expected"]
            
            try:
                results = retriever.invoke(query)[:k]
                all_content = " ".join([
                    doc.metadata.get('original_content', '') or doc.page_content 
                    for doc in results
                ])
                
                # 기대 키워드가 검색 결과에 포함되어 있는지 확인
                if expected in all_content:
                    correct += 1
            except Exception as e:
                pass
        
        accuracy = correct / total
        results_dict[name] = {"accuracy": accuracy, "correct": correct, "total": total}
    
    return results_dict

print("="*80)
print("Contextual Retrieval 성능 비교 (k=1, Top-1 정확도)")
print("="*80)

eval_results = evaluate_contextual_retrievers(retrievers_to_compare, test_queries, k=1)

for name, result in eval_results.items():
    status = "✅" if result["accuracy"] >= 0.5 else "❌"
    print(f"{status} {name:25s}: {result['accuracy']:.1%} ({result['correct']}/{result['total']})")

# 일반 vs Contextual 비교
print("\n" + "="*80)
print("일반 vs Contextual 비교 요약")
print("="*80)

normal_avg = sum([eval_results[k]["accuracy"] for k in ["일반 Embedding", "일반 BM25", "일반 Hybrid"]]) / 3
contextual_avg = sum([eval_results[k]["accuracy"] for k in ["Contextual Embedding", "Contextual BM25", "Contextual Hybrid"]]) / 3
best_result = eval_results["Contextual + Reranker"]["accuracy"]

print(f"일반 검색 평균 정확도:      {normal_avg:.1%}")
print(f"Contextual 검색 평균 정확도: {contextual_avg:.1%}")
print(f"Contextual + Reranker:      {best_result:.1%}")

if normal_avg > 0:
    improvement = ((contextual_avg - normal_avg) / normal_avg * 100)
    print(f"\n→ Contextual 방식이 일반 방식 대비 {improvement:.1f}% 향상")
else:
    print(f"\n→ 일반 검색 정확도가 0%이므로 비율 계산 불가")

Contextual Retrieval 성능 비교 (k=1, Top-1 정확도)
✅ 일반 Embedding             : 66.7% (4/6)
✅ 일반 BM25                  : 50.0% (3/6)
✅ 일반 Hybrid                : 66.7% (4/6)
✅ Contextual Embedding     : 83.3% (5/6)
✅ Contextual BM25          : 100.0% (6/6)
✅ Contextual Hybrid        : 83.3% (5/6)
✅ Contextual + Reranker    : 100.0% (6/6)

일반 vs Contextual 비교 요약
일반 검색 평균 정확도:      61.1%
Contextual 검색 평균 정확도: 88.9%
Contextual + Reranker:      100.0%

→ Contextual 방식이 일반 방식 대비 45.5% 향상


---

## 연습문제

다음 연습문제를 통해 Contextual Retrieval에 대한 이해를 확인해 보세요.

### 문제 1: 문맥 정보 생성 프롬프트

아래 코드의 빈칸을 채워 청크에 문맥 정보를 추가하는 프롬프트를 완성하세요.

In [29]:
from langchain_core.prompts import ChatPromptTemplate

# 문맥 정보 생성 프롬프트
context_prompt = ChatPromptTemplate.from_template(
    """다음은 전체 문서입니다:
<document>
{____}
</document>

다음은 문서의 일부 청크입니다:
<chunk>
{____}
</chunk>

이 청크가 전체 문서에서 어떤 맥락에 위치하는지 간단히 설명해주세요.
검색에 도움이 되도록 핵심 키워드와 주제를 포함해주세요.
응답은 2-3문장으로 작성하세요."""
)

# 힌트: 첫 번째 빈칸은 전체 문서, 두 번째 빈칸은 청크를 위한 변수명

### 문제 2: Contextual 청크 생성

아래 코드의 빈칸을 채워 문맥 정보가 포함된 청크를 생성하세요.

In [30]:
from langchain_core.documents import Document

# 원본 청크와 생성된 문맥 정보 (예시)
original_chunk = "Tesla는 2024년에 새로운 배터리 기술을 발표했습니다."
generated_context = "이 청크는 Tesla의 배터리 기술 발전에 관한 섹션에서 발췌되었습니다."

# Contextual 청크 생성
# 문맥 정보를 청크 앞에 추가하는 형식
contextual_content = f"[맥락] {____}\n\n{____}"  # 힌트: generated_context, original_chunk

# Document 객체 생성
contextual_doc = ____(
    page_content=contextual_content,
    metadata={"source": "Tesla_KR.md", "has_context": True}
)

print(f"Contextual 청크:\n{contextual_doc.page_content}")

NameError: name '____' is not defined

--- 
# **[실습]**

### 실습 목표

Contextual Retrieval을 실제 문서에 적용하여 검색 성능을 개선합니다.

### 난이도별 가이드

**기본 난이도:**
- 제공된 샘플 문서에 Contextual Retrieval 적용
- 일반 검색 vs Contextual 검색 성능 비교
- 3개 이상의 쿼리로 테스트

**중급 난이도:**
- 자체 문서(예: 기술 문서, 위키피디아) 활용
- Hybrid 검색 (Embedding + BM25) 구현
- 가중치 조합 실험 (0.3:0.7, 0.5:0.5, 0.7:0.3)

**고급 난이도:**
- 다국어 문서에 Contextual Retrieval 적용
- Reranker와 결합하여 최적 파이프라인 구성
- 검색 성능 지표 (HitRate, MRR) 측정 및 비교

# Hybrid + Reranker 기반 사내 공지 검색 시스템 

이 실습은 사내 그룹웨어 공지 데이터를 대상으로
**Embedding + BM25 + LLM Reranker**를 결합한 검색 시스템을 구축하는 과정입니다.

## 구조
1. Semantic Search (FAISS)
2. Keyword Search (BM25)
3. Hybrid Fusion (가중치 결합)
4. LLM Reranker (최종 정렬)

목표: 단순 검색이 아닌 “의미 기반 검색 + 정확도 개선”

## 사내 그룹웨어 공지 데이터 소개

본 실습에서 사용하는 데이터는 사내 그룹웨어에서 수집한 공지사항 데이터입니다.

### 데이터 수집 방식

- 사내 그룹웨어 게시판 API 기반 크롤링
- 게시판별 전체 목록 조회
- 각 게시글 상세 API 호출
- HTML 형태의 본문 데이터 텍스트 변환 처리

### 데이터 구성

각 문서는 다음 정보를 포함합니다:

- 공지 제목 (title)
- 본문 내용 (content)
- 게시판 종류 (board)
- 작성자 (writer)
- 작성일 (date)
- 문서 ID (id)

### 포함된 주요 공지 유형

- 윤리경영 / 윤리강령 공지
- 인사발령 및 조직 변경
- 휴가 / 복무 / 근태 규정
- 사내 교육
- 전사 공지 및 운영 안내
- 업무 프로세스 및 규정 문서

### 데이터 특징

- 문서 길이가 길고 형식이 다양함
- 동일 키워드라도 표현 방식이 다름
- 단순 키워드 검색으로는 정확한 검색이 어려움

### 문제 정의

기존 검색 방식은 다음 한계를 가짐:

- 키워드 매칭 중심 → 의미 검색 불가
- 긴 문서에서 핵심 정보 탐색 어려움
- 유사 문서 구분 성능 부족

=> 따라서 본 실습에서는
**RAG 기반 Hybrid Retrieval 시스템**으로 개선합니다.

`(1) 문서 준비 및 청킹`

자신의 문서를 로드하고 청크로 분할합니다.

- 긴 공지를 그대로 검색하면 성능이 떨어짐
- 그래서 chunk 단위로 분해
- metadata를 유지해서 “출처 추적 가능”
- chunk_overlap으로 문맥 유지

In [ ]:
# RAG 문서 구성 단계 1
# 목적: 원본 공지 데이터를 검색 가능한 작은 단위로 분해

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import pandas as pd

# ------------------------
# 1. 데이터 로드 (사내 공지 CSV)
# ------------------------
root = Path.cwd()

# 프로젝트 구조에서 faq_bot 폴더 위치 찾기
while not (root / "faq_bot").exists():
    root = root.parent

file_path = root / "faq_bot" / "companyboard.csv"
df = pd.read_csv(file_path)

# 결측값 처리 (공지 내용 없는 경우 방지)
df["content"] = df["content"].fillna("").astype(str)

# ------------------------
# 2. 텍스트 분할기 설정
# ------------------------
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,      # 문서 길이 제한
    chunk_overlap=150    # 문맥 유지용 overlap
)

# ------------------------
# 3. Document 생성 (메타데이터 유지)
# ------------------------
docs = []

for _, row in df.iterrows():

    # 문서 기본 정보 (검색 결과에서 활용)
    base_metadata = {
        "id": row["id"],
        "title": row["title"],     # 공지 제목
        "board": row["board"],     # 게시판 종류
        "writer": row["writer"],   # 작성자
        "date": row["date"]        # 작성일
    }

    content = row["content"]

    # 빈 문서 방지
    if not content:
        continue

    # 문서를 chunk 단위로 분리
    chunks = splitter.split_text(content)

    # chunk별 Document 생성
    for i, chunk in enumerate(chunks):

        docs.append(
            Document(
                page_content=chunk,
                metadata={
                    **base_metadata,
                    "chunk_id": i   # 문서 내부 위치 정보
                }
            )
        )

print("문서 로드 + 청킹 완료:", len(docs))

문서 로드 + 청킹 완료: 589


`(2) 컨텍스트 생성`

각 청크에 대해 맥락 설명을 생성합니다.

- 기존 문제: 문서 자체만으로는 의미 부족
- 해결: LLM이 “설명문 생성”
- 효과:
  embedding 품질 상승,
  검색 정확도 증가

In [ ]:
# RAG 성능 개선 단계
# 각 chunk에 "검색용 의미 설명(Context)" 추가

from langchain_openai import ChatOpenAI

# LLM 설정 (저비용 빠른 모델)
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.1  # 안정적인 설명 생성
)

# ------------------------
# Context 생성 함수
# ------------------------
def make_context(doc):

    # LLM에게 문서 의미 요약 요청
    prompt = f"""
다음은 사내 그룹웨어 공지의 일부이다.

이 내용이 어떤 주제/상황인지
검색 최적화를 위해 한 줄로 설명해라.

조건:
- 키워드 중심
- 1문장
- 공지 유형 포함

본문:
{doc.page_content}
"""

    context = llm.invoke(prompt).content

    # 기존 문서 앞에 context 추가
    doc.page_content = f"""[CONTEXT]
{context}

[TEXT]
{doc.page_content}
"""

    return doc


# 모든 chunk에 context 생성 적용
context_docs = []

for d in docs:
    context_docs.append(make_context(d))

print("컨텍스트 생성 완료:", len(context_docs))

컨텍스트 생성 완료: 589


`(3) 하이브리드 검색 및 평가`

Embedding + BM25 + Reranker 파이프라인을 구성하고 성능을 평가합니다.

## 1단계: Hybrid Search (FAISS + BM25)

이 단계에서는 두 가지 검색 방식을 결합하여 후보 문서를 생성합니다.

### ✔ FAISS (Semantic Search)
- 문장의 의미 기반 검색
- 유사한 의미를 가진 문서 탐색

### ✔ BM25 (Keyword Search)
- 단어 기반 정확한 매칭
- 특정 키워드 포함 문서 탐색

### ✔ 결합 방식
- 두 결과를 합쳐 후보 문서 생성
- embedding 중심 + keyword 보조 구조

## 2단계: LLM Reranker

Hybrid Search로 가져온 후보 문서들을
LLM을 이용해 다시 정렬합니다.

### ✔ 역할
- 문서와 질문의 실제 의미적 관련성 평가
- 0~10 점으로 점수화
- 최종 순위 재정렬

### ✔ 효과
- 단순 유사도 기반 검색의 한계 보완
- 실제 “의미 기준 검색 결과” 생성

In [38]:
# Hybrid Retrieval 시스템
# 목적: semantic + keyword + LLM ranking 결합

from rank_bm25 import BM25Okapi
import numpy as np
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

root = Path.cwd()
while not (root / "faq_bot").exists():
    root = root.parent

db_path = root / "faq_bot" / "gw_notice_db"

# ------------------------
# FAISS Vector DB 로드
# ------------------------
db = FAISS.load_local(
    db_path,
    OpenAIEmbeddings(model="text-embedding-3-large"),
    allow_dangerous_deserialization=True
)

db_context = db  # semantic search용 DB

# ------------------------
# BM25 (키워드 기반 검색)
# ------------------------
corpus = [d.page_content for d in docs]
tokenized_corpus = [c.split() for c in corpus]

bm25 = BM25Okapi(tokenized_corpus)

# ------------------------
# Hybrid Search
# ------------------------
def hybrid_search(query, k=8, alpha=0.7):

    # 1. Semantic search (FAISS)
    emb_docs = db_context.similarity_search(query, k=10)

    # 2. Keyword search (BM25)
    scores = bm25.get_scores(query.split())

    # 상위 5개만 사용 (noise 제거)
    top_idx = np.argsort(scores)[::-1][:5]
    bm_docs = [docs[i] for i in top_idx]

    merged = {}

    # semantic weight (중요)
    for d in emb_docs:
        merged[d.metadata["id"]] = {
            "doc": d,
            "score": 0.8
        }

    # keyword 보조 보정
    for d in bm_docs:
        if d.metadata["id"] in merged:
            merged[d.metadata["id"]]["score"] += 0.2
        else:
            merged[d.metadata["id"]] = {
                "doc": d,
                "score": 0.2
            }

    ranked = sorted(
        merged.values(),
        key=lambda x: x["score"],
        reverse=True
    )

    return [r["doc"] for r in ranked[:k]]


def rerank(query, docs):

    results = []

    for d in docs:

        prompt = f"""
Query: {query}

Document:
{d.page_content}

0~10점으로 관련성 평가.
숫자만 출력.
"""

        score = float(llm.invoke(prompt).content.strip())

        results.append((d, score))

    results.sort(key=lambda x: x[1], reverse=True)

    # 점수까지 같이 반환
    return results

In [40]:
def debug_all(query):

    print("\n" + "="*80)
    print(f"QUERY: {query}")
    print("="*80)

    # FAISS
    emb_docs = db_context.similarity_search(query, k=5)
    print("\nFAISS 결과")
    for d in emb_docs:
        print("-", d.metadata["title"])

    # BM25
    scores = bm25.get_scores(query.split())
    top_idx = np.argsort(scores)[::-1][:5]
    bm_docs = [docs[i] for i in top_idx]

    print("\nBM25 결과")
    for d in bm_docs:
        print("-", d.metadata["title"])

    # Hybrid
    hybrid_docs = hybrid_search(query, k=5)
    print("\nHybrid 결과")
    for d in hybrid_docs:
        print("-", d.metadata["title"])

    # Rerank
    reranked = rerank(query, hybrid_docs)
    print("\nRerank 결과")
    for d, score in reranked:
        print("-", d.metadata["title"], score)

debug_all("규정 관련 공지")


QUERY: 규정 관련 공지

FAISS 결과
- 사규관리규정
- 복무규정
- [당사 임직원 정보관리, 주식매매 관련 규정 및 유의사항]
- 인사규정
- 인사고과규정

BM25 결과
- [당사 임직원 정보관리, 주식매매 관련 규정 및 유의사항]
- 정보보안 규정
- 기숙사 이용 규정
- 급여(연봉제)규정(2026)
- 문서보관및보존규정

Hybrid 결과
- [당사 임직원 정보관리, 주식매매 관련 규정 및 유의사항]
- 정보보안 규정
- 사규관리규정
- 복무규정
- 인사규정

Rerank 결과
- [당사 임직원 정보관리, 주식매매 관련 규정 및 유의사항] 10.0
- 정보보안 규정 10.0
- 인사규정 10.0
- 사규관리규정 2.0
- 복무규정 2.0


## 3단계: 최종 결과 확인 (Top1 Preview)

최종적으로 선택된 문서의 내용을 확인하여
검색 결과의 실제 품질을 검증합니다.

### ✔ 확인 목적
- 제목 정확도
- 본문 관련성
- 실제 사용자 관점 적합성

👉 검색 시스템의 UX 품질 점검 단계

### 테스트 쿼리 설정

검색 시스템의 성능을 확인하기 위해 대표적인 사내 공지 유형에 대한 질문을 정의합니다.

- 윤리경영 / 윤리강령
- 휴가 및 복무 규정
- 사내 교육 안내
- 인사발령 공지

다양한 유형의 질문으로 검색 성능을 검증합니다.

In [23]:
# ------------------------
# 테스트 쿼리
# ------------------------
test_queries = [
    "윤리경영 공지",
    "휴가 신청 규정",
    "교육 안내",
    "인사발령 내용"
]

# ------------------------
# 결과 출력
# ------------------------
for q in test_queries:

    print("\n" + "="*80)
    print(f"질문: {q}")
    print("="*80)

    # 1. Hybrid 검색 결과
    hybrid_results = hybrid_search(q, k=8)

    print("\n[1] Hybrid Search 결과 (FAISS + BM25)")
    for i, d in enumerate(hybrid_results, 1):
        print(f"{i}. {d.metadata['title']}")

    # 2. Rerank 결과
    reranked_results = rerank(q, hybrid_results)

    print("\n[2] Rerank 결과 (LLM 재정렬)")
    for i, d in enumerate(reranked_results[:5], 1):
        print(f"{i}. {d.metadata['title']}")

    # 3. 본문 일부 확인 (가독성용)
    print("\n[3] Top 1 문서 미리보기")
    top_doc = reranked_results[0]
    print(f"제목: {top_doc.metadata['title']}")
    print(f"내용 일부:\n{top_doc.page_content[:300]}")


질문: 윤리경영 공지

[1] Hybrid Search 결과 (FAISS + BM25)
1. 하이비젼시스템 윤리경영 문화 정착 관련 공지
2. [공지] "추석 명절 선물 수수 금지" 
3. 윤리강령
4. 사내 윤리강령 제정 및 실천구호 공모 안내
5. [당사 임직원 정보관리, 주식매매 관련 규정 및 유의사항]
6. [윤리강령] 제정에 따른 실천구호 선정 공지
7. 이사회운영규정
8. 주말식당 이용 및 국내출장 식대 한도변경, 동아리 지원금에 대한 공지

[2] Rerank 결과 (LLM 재정렬)
1. 하이비젼시스템 윤리경영 문화 정착 관련 공지
2. [공지] "추석 명절 선물 수수 금지" 
3. [윤리강령] 제정에 따른 실천구호 선정 공지
4. 사내 윤리강령 제정 및 실천구호 공모 안내
5. 윤리강령

[3] Top 1 문서 미리보기
제목: 하이비젼시스템 윤리경영 문화 정착 관련 공지
내용 일부:
안녕하십니까
하이비젼시스템
청렴한
윤리경영
문화
정착을
위하여
평소에도
금품
수수나
향응
·
접대
등의
비윤리
행위를
일체
금지하고
있습니다
.
거래처에서
회사로
배송하는
공개적이며
대표성격의
경우를
제외하고
임직원
개인
자택에서
선물
(
상품권
등
)
을
수수할
경우
회사
규정에
따라
불이익
을
받을
수
있으니
이점
유의하여
주시기
바랍니다
.
경영의
기본조건
윤리경영은
선택이
아닌
필수
조건임을
하이메이트
구성원
여러분들께서
양지하시어
이해충돌을
사전에
방지하여
청렴한
준법경영을
실천하고
사회적
책임을
다하는
기업이
될
수
있도

질문: 휴가 신청 규정

[1] Hybrid Search 결과 (FAISS + BM25)
1. [공지] 2024 장기휴가 현황
2. 기숙사 이용 규정
3. 취업규칙(24.07.01)
4. 복무규정
5. 업무분장규정(2026)
6. 사원채용규정
7. 급여(연봉제)규정(2026)
8. [안내] 장기휴가제도 안내(리마인드)

[2] Rerank 결과 (LLM 재정렬)
1. 복무규정
2. [공지] 2024 장기휴가 현황
3. 취업규칙(24.0

In [ ]:
import gradio as gr

# =========================================================
# 1. Chat 함수 정의
# =========================================================
# Gradio에서 입력을 받아서 RAG 검색 결과 반환
def chatbot(message, history):

    # 기존 Hybrid + Reranker 검색 함수 호출
    results = search(message)

    # 결과 포맷팅 (사용자 보기 좋게)
    answer = ""

    for i, doc in enumerate(results, 1):

        answer += f"""
[{i}] {doc.metadata['title']}
- 게시판: {doc.metadata['board']}
- 작성자: {doc.metadata['writer']}
- 작성일: {doc.metadata['date']}

내용:
{doc.page_content[:300]}

----------------------------------------
"""

    return answer


# =========================================================
# 2. Gradio UI 설정
# =========================================================
demo = gr.ChatInterface(
    fn=chatbot,
    title="📄 사내 그룹웨어 공지 RAG 검색 시스템",
    description="""
Hybrid Retrieval 기반 검색 시스템입니다.

- 의미 기반 검색 (Semantic Search)
- 키워드 검색 (BM25)
- LLM 재정렬 (Reranker)

사내 공지 데이터를 기반으로 질문에 답변합니다.
"""
)

# =========================================================
# 3. 실행
# =========================================================
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


c:\Users\HVS\modu_llm7\etf-bot\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\HVS\modu_llm7\etf-bot\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\HVS\modu_llm7\etf-bot\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\HVS\modu_llm7\etf-bot\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_

In [ ]:
import gradio as gr

def chatbot(message, history):

    results = search(message)

    output = ""

    for i, (doc, score) in enumerate(results, 1):

        output += f"""
==================================================
[{i}] RANK SCORE: {score}

제목: {doc.metadata['title']}
게시판: {doc.metadata['board']}
작성자: {doc.metadata['writer']}
작성일: {doc.metadata['date']}

선택 이유 (LLM 판단 기준)
→ 질문과 의미적으로 높은 관련성으로 판단됨

본문 일부:
{doc.page_content[:300]}

==================================================
"""

    return output


demo = gr.ChatInterface(
    fn=chatbot,
    title="Hybrid RAG 공지 검색 시스템 (Explainable)",
    description="FAISS + BM25 + LLM Reranker 기반 검색 + 선택 근거 표시"
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


c:\Users\HVS\modu_llm7\etf-bot\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\HVS\modu_llm7\etf-bot\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\HVS\modu_llm7\etf-bot\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\HVS\modu_llm7\etf-bot\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_